# 생성 경로 지연: Gemma 4 31B, Google AI Studio 무료 대 OpenRouter

생성 기본 경로를 정하려고 같은 모델(Gemma 4 31B)을 두 경로로 불러, 사고 과정을 끌 수 있는지와 답변 첫 토큰까지의 시간을 잰다. 질의는 지어낸 질문 10개와 지어낸 근거다(AI Hub 데이터가 아님). 프롬프트는 `configs/prompt/v1.yaml` 형식을 따른다.

## 파라미터 확인 (공식 문서, 2026-10-19)

- Google: [Gemma on Gemini API](https://ai.google.dev/gemma/docs/core/gemma_on_gemini_api)는 Gemma 4의 사고 과정을 `thinkingLevel` `"high"`(켬)·`"minimal"`(끔)으로 켜고 끈다고 적는다. 원본 API는 `generationConfig.thinkingConfig.thinkingLevel`, OpenAI 호환 엔드포인트는 `extra_body.google.thinking_config.thinking_level`이다([OpenAI compatibility](https://ai.google.dev/gemini-api/docs/openai)는 Gemma를 다루지 않는다). D-11에서 400이 난 `reasoning_effort`, `thinking_budget`은 이 모델의 파라미터가 아니다
- OpenRouter: 모델 목록 API(`/api/v1/models`)에서 `google/gemma-4-31b-it`의 `reasoning`은 `{mandatory: false, default_enabled: false}`다. 끄는 값은 [reasoning tokens](https://openrouter.ai/docs/use-cases/reasoning-tokens)의 `reasoning.effort: "none"`이다. 제공자 15곳(fp4·fp8·bf16), 가격은 제공자마다 다르다(DeepInfra turbo 입력 $0.135·출력 $0.51, 100만 토큰당)

In [1]:
import json
import time
from pathlib import Path

import httpx
import pandas as pd
import yaml

In [2]:
ROOT = Path.cwd().parent
RAW = ROOT / "data" / "runs" / "generate" / "route_latency" / "gemma4_2026-10-19.jsonl"
SMOKE = RAW.with_name("gemma4_2026-10-19_smoke.jsonl")

## 측정 코드

아래 정의를 스크립트로 실행했다(2026-10-19, `variants` 5개 × 질문 10개, 경로 교대로 호출, Google 호출 뒤 4초 대기). 원자료는 `RAW`에 있다. API를 다시 부르지 않도록 노트북에서는 정의만 두고 실행하지 않는다. 스트리밍으로 첫 토큰과 답변 첫 토큰(사고 과정 뒤)을 잰다. Google OpenAI 호환은 사고 과정을 `content` 앞 `<thought>…</thought>`로, 원본 API는 `thought: true` 파트로, OpenRouter는 `reasoning` 필드로 낸다

In [3]:
import json
import sys
import time
from pathlib import Path

import httpx
import yaml

ENV = dict(l.split("=", 1) for l in (ROOT / ".env").read_text(encoding="utf-8").splitlines()
           if "=" in l and not l.lstrip().startswith("#"))
ENV = {k.strip(): v.strip().strip('"').strip("'") for k, v in ENV.items()}
PROMPT = yaml.safe_load((ROOT / "configs/prompt/v1.yaml").read_text(encoding="utf-8"))
PARAMS = {"temperature": 0.0, "top_p": 1.0, "max_tokens": 1024}

# 가상 질문과 지어낸 근거(실제 회의록이 아님)
CASES = [
    ("2023년 국정감사에서 교육부의 디지털 교과서 도입 일정에 대해 어떤 지적이 있었나요?",
     ["2023-10-12 교육위원회 국정감사 제1차", "김 위원: 디지털 교과서를 2025년에 전면 도입한다는 계획은 교사 연수가 충분하지 않은 상황에서 무리하다고 봅니다. 시범학교 결과도 아직 공개되지 않았습니다.",
      "2023-10-12 교육위원회 국정감사 제1차", "장관: 2025년에는 수학·영어·정보 과목부터 단계적으로 도입하고 교사 연수는 올해 하반기에 3만 명 규모로 진행하겠습니다."]),
    ("전세사기 피해자 지원 특별법 논의에서 보증금 선지급 방안에 대한 정부 입장은 무엇이었나요?",
     ["2023-05-10 국토교통위원회 법안심사소위원회", "차관: 보증금을 국가가 먼저 지급하는 방식은 다른 사기 피해와의 형평성 문제가 있어 수용하기 어렵습니다. 대신 경매 유예와 우선매수권을 드리겠습니다."]),
    ("의대 정원 확대와 관련해 지역의사제 도입을 주장한 발언 내용을 알려 주세요.",
     ["2024-02-20 보건복지위원회 전체회의", "이 위원: 정원만 늘리면 수도권 쏠림이 심해집니다. 지역에서 10년 이상 의무 복무하는 지역의사제를 함께 도입해야 합니다."]),
    ("반도체 특별법에서 연구개발 인력의 주 52시간 예외 적용에 대해 어떤 우려가 나왔나요?",
     ["2024-11-28 산업통상자원중소벤처기업위원회 소위원회", "박 위원: 연구개발 인력에만 예외를 둔다고 하지만 현장에서는 생산직까지 확대 적용될 우려가 있습니다. 노사 합의 요건을 분명히 해야 합니다."]),
    ("기후위기 대응을 위한 탄소중립 기본법 개정 논의에서 2030년 감축 목표는 어떻게 다뤄졌나요?",
     ["2022-09-15 환경노동위원회 전체회의", "장관: 2030년 국가 온실가스 감축 목표 40%는 유지하되 산업 부문 감축률을 14.5%에서 11.4%로 조정하는 방안을 검토하고 있습니다."]),
    ("저출생 대책으로 논의된 육아휴직 급여 인상안의 구체적인 금액은 얼마였나요?",
     ["2024-06-18 보건복지위원회 전체회의", "차관: 육아휴직 급여 상한을 월 150만 원에서 첫 3개월 250만 원으로 올리고, 이후 3개월은 200만 원으로 하는 방안을 마련했습니다."]),
    ("국가채무 증가와 관련해 재정준칙 법제화에 반대한 이유는 무엇이었나요?",
     ["2023-04-25 기획재정위원회 경제재정소위원회", "최 위원: 재정준칙을 법률로 묶으면 경기 침체기에 필요한 확장 재정을 할 수 없습니다. 시행령 수준에서 유연하게 운용해야 합니다."]),
    ("농산물 가격안정제 도입 논의에서 양곡관리법 개정안과의 차이를 설명한 발언이 있나요?",
     ["2024-03-07 농림축산식품해양수산위원회 전체회의", "위원장: 가격안정제는 기준가격과 시장가격의 차액을 보전하는 방식이고, 양곡관리법 개정안은 초과 생산량을 정부가 의무 매입하는 방식이라 재정 부담 구조가 다릅니다."]),
    ("인공지능 기본법 심사에서 고위험 인공지능의 정의를 두고 어떤 의견 차이가 있었나요?",
     ["2024-12-03 과학기술정보방송통신위원회 법안심사소위원회", "정 위원: 고위험 인공지능을 영역으로만 정의하면 규제 범위가 지나치게 넓어집니다. 사람의 생명·안전·기본권에 중대한 영향을 미치는 경우로 한정해야 합니다."]),
    ("지방소멸 대응기금의 배분 기준에 대해 행정안전부는 어떻게 답변했나요?",
     ["2023-10-17 행정안전위원회 국정감사", "장관: 지방소멸 대응기금은 인구감소지역 89곳에 매년 1조 원을 배분하며, 투자계획 평가 결과에 따라 지역별로 차등 지급하고 있습니다."]),
]


def messages(query: str, ev: list[str]) -> list[dict]:
    blocks = [f"[{i // 2 + 1}] {ev[i]}\n{ev[i + 1]}" for i in range(0, len(ev), 2)]
    return [{"role": "system", "content": PROMPT["system"]},
            {"role": "user", "content": PROMPT["user"].format(contexts="\n\n".join(blocks), query=query)}]


def sse(resp):
    for line in resp.iter_lines():
        if line.startswith("data:"):
            data = line[5:].strip()
            if data and data != "[DONE]":
                yield json.loads(data)


def openai_stream(url: str, key: str, body: dict, thought_in_content: bool) -> dict:
    """OpenAI 호환 스트리밍. Google은 사고 과정을 content 앞 <thought>로, OpenRouter는 reasoning으로 낸다."""
    t0 = time.perf_counter()
    first = answer = None
    content, reasoning, usage, provider = "", "", None, None
    headers = {"Authorization": f"Bearer {key}"}
    with httpx.stream("POST", url, json=body, headers=headers, timeout=180) as r:
        if r.status_code != 200:
            return {"error": r.status_code, "body": r.read().decode()[:500]}
        for ch in sse(r):
            now = time.perf_counter() - t0
            provider = ch.get("provider", provider)
            if ch.get("usage"):
                usage = ch["usage"]
            for c in ch.get("choices") or []:
                d = c.get("delta") or {}
                rs = d.get("reasoning") or d.get("reasoning_content") or ""
                if rs:
                    reasoning += rs
                    first = first or now
                if d.get("content"):
                    first = first or now
                    content += d["content"]
                    if answer is None:
                        if not thought_in_content or not content.lstrip().startswith("<thought>"[:len(content.lstrip())]):
                            answer = now
                        elif "</thought>" in content and content.split("</thought>", 1)[1].strip():
                            answer = now
    total = time.perf_counter() - t0
    thought = "<thought>" in content or bool(reasoning)
    return {"first": first, "answer_first": answer, "total": total, "thought": thought,
            "thought_chars": len(reasoning) + (len(content.split("</thought>")[0]) if "<thought>" in content else 0),
            "answer_chars": len(content.split("</thought>")[-1].strip()), "usage": usage, "provider": provider}


def native_stream(level: str | None, msgs: list[dict]) -> dict:
    """Gemini API 원본(generateContent) 스트리밍. 사고 과정 파트는 thought: true로 온다."""
    url = ("https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:"
           f"streamGenerateContent?alt=sse&key={ENV['GOOGLE_API_KEY']}")
    gen = {"temperature": 0.0, "topP": 1.0, "maxOutputTokens": 1024}
    if level:
        gen["thinkingConfig"] = {"thinkingLevel": level, "includeThoughts": True}
    body = {"systemInstruction": {"parts": [{"text": msgs[0]["content"]}]},
            "contents": [{"role": "user", "parts": [{"text": msgs[1]["content"]}]}],
            "generationConfig": gen}
    t0 = time.perf_counter()
    first = answer = None
    tchars = achars = 0
    usage = None
    with httpx.stream("POST", url, json=body, timeout=180) as r:
        if r.status_code != 200:
            return {"error": r.status_code, "body": r.read().decode()[:500]}
        for ch in sse(r):
            now = time.perf_counter() - t0
            usage = ch.get("usageMetadata", usage)
            for cand in ch.get("candidates") or []:
                for p in (cand.get("content") or {}).get("parts") or []:
                    if not p.get("text"):
                        continue
                    first = first or now
                    if p.get("thought"):
                        tchars += len(p["text"])
                    else:
                        achars += len(p["text"])
                        answer = answer or now
    return {"first": first, "answer_first": answer, "total": time.perf_counter() - t0,
            "thought": tchars > 0, "thought_chars": tchars, "answer_chars": achars, "usage": usage}


def run(variant: str, msgs: list[dict]) -> dict:
    g_url = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
    o_url = "https://openrouter.ai/api/v1/chat/completions"
    base = {"messages": msgs, "stream": True, **PARAMS}
    if variant == "google_compat_default":
        return openai_stream(g_url, ENV["GOOGLE_API_KEY"], {"model": "models/gemma-4-31b-it", **base,
                             "stream_options": {"include_usage": True}}, True)
    if variant == "google_compat_minimal":
        return openai_stream(g_url, ENV["GOOGLE_API_KEY"], {"model": "models/gemma-4-31b-it", **base,
                             "stream_options": {"include_usage": True},
                             "extra_body": {"google": {"thinking_config": {"thinking_level": "minimal"}}}}, True)
    if variant == "google_native_default":
        return native_stream(None, msgs)
    if variant == "google_native_minimal":
        return native_stream("minimal", msgs)
    if variant == "openrouter_default":
        return openai_stream(o_url, ENV["OPENROUTER_API"], {"model": "google/gemma-4-31b-it", **base,
                             "usage": {"include": True}}, False)
    if variant == "openrouter_none":
        return openai_stream(o_url, ENV["OPENROUTER_API"], {"model": "google/gemma-4-31b-it", **base,
                             "usage": {"include": True}, "reasoning": {"effort": "none"}}, False)
    raise ValueError(variant)



## 집계

시간은 오류(500)를 뺀 호출로 계산한다. 표본이 10개라 p95 대신 최대값을 본다. 답변 없음은 사고 과정만 내고 끝난 호출이다.

In [4]:
d = pd.DataFrame([json.loads(x) for x in RAW.open(encoding="utf-8")])
d["err"] = d["error"].notna()
d["no_answer"] = d["answer_first"].isna() & ~d["err"]
ok = d[~d["err"]].groupby("variant", sort=False)
g = d.groupby("variant", sort=False)
pd.DataFrame({"n": g.size(), "error": g["err"].sum(), "no_answer": g["no_answer"].sum(),
              "thought": g["thought"].sum(),
              "answer_first_mean": ok["answer_first"].mean(), "answer_first_p50": ok["answer_first"].median(),
              "answer_first_max": ok["answer_first"].max(),
              "total_mean": ok["total"].mean(), "total_max": ok["total"].max()}).round(2)

,n,error,no_answer,thought,answer_first_mean,answer_first_p50,answer_first_max,total_mean,total_max
variant,,,,,,,,,
google_compat_default,10,4,2,6,47.56,46.57,69.03,42.19,70.40
google_compat_minimal,10,3,0,0,32.75,36.87,46.54,40.26,72.23
google_native_minimal,10,3,0,0,24.83,28.17,31.60,28.61,47.61
openrouter_default,10,0,0,0,1.52,1.29,3.77,4.83,13.68
openrouter_none,10,0,0,0,1.43,1.03,4.78,4.63,8.41


In [5]:
orr = d[d["variant"].str.startswith("openrouter")]
usage = pd.DataFrame(list(orr["usage"]))
print("제공자", orr["provider"].value_counts().to_dict())
print("입력·출력 토큰", int(usage["prompt_tokens"].sum()), int(usage["completion_tokens"].sum()),
      "비용 $", round(usage["cost"].sum(), 5))
print(d[d["err"]][["variant", "case", "error"]].to_string(index=False))

제공자 {'DeepInfra': 20}
입력·출력 토큰 5032 952 비용 $ 0.00154
              variant  case  error
google_compat_default     0  500.0
google_compat_default     2  500.0
google_native_minimal     2  500.0
google_native_minimal     3  500.0
google_compat_default     4  500.0
google_compat_minimal     4  500.0
google_compat_default     5  500.0
google_compat_minimal     5  500.0
google_native_minimal     5  500.0
google_compat_minimal     6  500.0


## 동작 확인 (질문 1개, 본 측정 전)

`SMOKE`는 질문 1개로 여섯 설정을 부른 기록이다. 원본 API 기본 설정은 100초 동안 사고 토큰 10개만 내고 끝났고, 원본 API `minimal`은 500이었다. 같은 질문을 원본 API로 스트리밍 없이 다시 부르면 기본 설정 500(1.5초), `minimal` 200(38.6초, `thoughtsTokenCount` 없음, 사고 파트 빈 문자열), `high` 200(31.9초, `thoughtsTokenCount` 364)이었다(출력은 보관하지 않음).

In [6]:
pd.DataFrame([json.loads(x) for x in SMOKE.open(encoding="utf-8")])[
    ["variant", "first", "answer_first", "total", "thought", "error"]].round(2)

,variant,first,answer_first,total,thought,error
0,google_compat_default,14.56,NaN,19.75,True,NaN
1,google_compat_minimal,19.61,19.61,20.69,False,NaN
2,google_native_default,62.32,NaN,100.14,True,NaN
3,google_native_minimal,NaN,NaN,NaN,NaN,500.0
4,openrouter_default,1.17,1.17,4.28,False,NaN
5,openrouter_none,1.09,1.09,4.02,False,NaN


## 정리

- 사고 과정은 두 경로 모두 끌 수 있다. Google은 `thinking_level`/`thinkingLevel` `minimal`, OpenRouter는 기본값이 꺼짐이고 `reasoning.effort: "none"`으로 명시할 수 있다
- Google 무료 경로는 사고 과정을 꺼도 답변 첫 토큰이 평균 25\~33초(최대 32\~47초)이고, 설정마다 10번 중 3\~4번이 500이었다. 지연은 사고 과정보다 무료 경로 자체에서 온다(추론, 하루 한 시간대의 측정)
- OpenRouter는 답변 첫 토큰 중앙값 1.0\~1.3초, 최대 3.8\~4.8초, 오류 0이다. 20번 모두 DeepInfra로 갔다
- 입력이 약 300토큰으로 실제 생성(근거 5개, 약 2,500토큰)보다 짧다. 실제 입력에서 첫 토큰은 조금 늘 수 있다(재지 않음)